In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping # Import EarlyStopping
import time
import matplotlib.pyplot as plt
import pandas as pd

# 1. TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. HÀM TẠO MÔ HÌNH
def build_basic():
    return models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'), layers.Flatten(),
        layers.Dense(64, activation='relu'), layers.Dense(10, activation='softmax')
    ])

def build_improved(drop=0.3):
    return models.Sequential([
        layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(32,32,3)),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)), layers.Dropout(drop),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2,2)), layers.Dropout(drop),
        layers.Flatten(), layers.Dense(128, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(drop),
        layers.Dense(10, activation='softmax')
    ])

# 3. HÀM HUẤN LUYỆN (Rút gọn code lặp)
def train_model(model, optimizer, name, epochs=15, patience=5, monitor='val_loss', mode='min'):
    print(f"\n========== ĐANG TRAIN: {name} ==========")
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Early Stopping callback
    early_stopping = EarlyStopping(monitor=monitor, patience=patience, mode=mode, restore_best_weights=True)
    callbacks = [early_stopping]

    start = time.time()
    hist = model.fit(x_train, y_train, epochs=epochs, validation_data=(x_test, y_test), batch_size=64, verbose=0, callbacks=callbacks) # Set verbose to 0 to suppress epoch-by-epoch output
    time_taken = time.time() - start

    final_val_accuracy = hist.history['val_accuracy'][-1]
    final_val_loss = hist.history['val_loss'][-1]
    final_train_loss = hist.history['loss'][-1]

    print(f"-> Xong {name} | Thᄔi gian: {time_taken:.1f}s | Val Acc: {final_val_accuracy*100:.2f}% | Val Loss: {final_val_loss:.4f} | Train Loss: {final_train_loss:.4f}")
    return hist, final_val_accuracy, final_val_loss, final_train_loss

# 4. CHẠY THỰC NGHIỆM
if tf.test.gpu_device_name(): print("Đã nhận GPU!")
else: print("Cảnh báo: Đang chạy bằng CPU, sẽ khá chậm.")

metrics_data = []

h1, val_acc1, val_loss1, train_loss1 = train_model(build_basic(), optimizers.SGD(learning_rate=0.01), "B1: Cơ bản (SGD)")
metrics_data.append({
    'Model': 'B1 (SGD)',
    'Validation Accuracy': val_acc1,
    'Validation Loss': val_loss1,
    'Training Loss': train_loss1
})

h2, val_acc2, val_loss2, train_loss2 = train_model(build_improved(), optimizers.SGD(learning_rate=0.01, momentum=0.9), "B2: Cải tiến (SGD+Momentum)")
metrics_data.append({
    'Model': 'B2 (Momentum)',
    'Validation Accuracy': val_acc2,
    'Validation Loss': val_loss2,
    'Training Loss': train_loss2
})

h3, val_acc3, val_loss3, train_loss3 = train_model(build_improved(), 'adam', "B2: Cải tiến (Adam)")
metrics_data.append({
    'Model': 'B2 (Adam)',
    'Validation Accuracy': val_acc3,
    'Validation Loss': val_loss3,
    'Training Loss': train_loss3
})

# Display consolidated metrics
print("\n========== BẢNG TÓM TẮT KẾT QUẢ ==========")
metrics_df = pd.DataFrame(metrics_data)
metrics_df['Validation Accuracy'] = metrics_df['Validation Accuracy'].apply(lambda x: f'{x*100:.2f}%')
display(metrics_df)

# 5. VẼ BIỂU ĐỒ SO SÁNH (Sử dụng màu đỏ nổi bật cho đường line đầu tiên)
plt.figure(figsize=(14, 5))
metrics = [('val_accuracy', 'Validation Accuracy'), ('loss', 'Training Loss')]

for i, (metric, title) in enumerate(metrics):
    plt.subplot(1, 2, i+1)
    plt.plot(h1.history[metric], 'r-o', label='B1 (SGD)')
    plt.plot(h2.history[metric], 'g-s', label='B2 (Momentum)')
    plt.plot(h3.history[metric], 'b-^', label='B2 (Adam)')
    plt.title(title)
    plt.xlabel('Epochs')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

  1941504/170498071 ━━━━━━━━━━━━━━━━━━━━ 3:58:37 85us/step

## Tối ưu Model B2

Bạn có thể điều chỉnh các tham số `dropout_rate` và `learning_rate` dưới đây để tối ưu hóa hiệu suất của Model B2.

In [ ]:
#@title Tùy chỉnh tham số cho Model B2 (Adam)
dropout_rate = 0.5 #@param {type: "slider", min:0.1, max:0.6, step:0.1}
learning_rate = 0.0019 #@param {type: "slider", min:0.0001, max:0.01, step:0.0001}

print(f"\n========== ĐANG TRAIN: B2: Cải tiến (Adam) với Dropout={dropout_rate}, Learning Rate={learning_rate} ==========")

# Re-build and compile the improved model with new parameters
model_b2_optimized = build_improved(drop=dropout_rate)
optimizer_adam_optimized = optimizers.Adam(learning_rate=learning_rate)
model_b2_optimized.compile(optimizer=optimizer_adam_optimized, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start = time.time()
hist_optimized = model_b2_optimized.fit(x_train, y_train, epochs=15, validation_data=(x_test, y_test), batch_size=64, verbose=2)
time_taken_optimized = time.time() - start

final_val_accuracy_optimized = hist_optimized.history['val_accuracy'][-1]
final_val_loss_optimized = hist_optimized.history['val_loss'][-1]
final_train_loss_optimized = hist_optimized.history['loss'][-1]

print(f"\n-> Xong B2 tối ưu | Thời gian: {time_taken_optimized:.1f}s | Val Acc: {final_val_accuracy_optimized*100:.2f}% | Val Loss: {final_val_loss_optimized:.4f} | Train Loss: {final_train_loss_optimized:.4f}")

# Display optimized metrics
optimized_metrics = pd.DataFrame([{
    'Model': 'B2 (Adam) Optimized',
    'Validation Accuracy': final_val_accuracy_optimized,
    'Validation Loss': final_val_loss_optimized,
    'Training Loss': final_train_loss_optimized
}])
optimized_metrics['Validation Accuracy'] = optimized_metrics['Validation Accuracy'].apply(lambda x: f'{x*100:.2f}%')

print("\n========== KẾT QUẢ TỐI ƯU MODEL B2 ==========")
display(optimized_metrics)


## Biểu đồ so sánh hiệu suất các Model (bao gồm Model B2 tối ưu)

In [ ]:
plt.figure(figsize=(14, 5))
metrics_to_plot = [('val_accuracy', 'Validation Accuracy'), ('loss', 'Training Loss')]

for i, (metric, title) in enumerate(metrics_to_plot):
    plt.subplot(1, 2, i+1)
    plt.plot(h1.history[metric], 'r-o', label='B1 (SGD)')
    plt.plot(h2.history[metric], 'g-s', label='B2 (Momentum)')
    plt.plot(h3.history[metric], 'b-^', label='B2 (Adam - Original)')
    plt.plot(hist_optimized.history[metric], 'm-x', label=f'B2 (Adam - Optimized DR={dropout_rate}, LR={learning_rate})') # Add the optimized model
    plt.title(title)
    plt.xlabel('Epochs')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()